# Memory Exercises

## Memory in LangGraph - Quick Checks

These short questions are designed to check your understanding of the memory concepts demonstrated.

### Quick Check Questions

1. **Understanding Persistence**
   - What role does a checkpointer (e.g., `InMemorySaver`) play in a LangGraph workflow? Why is it necessary for persisting state across runs?

2. **Thread Isolation**
   - If two users interact with the same workflow at the same time, how does the system ensure that their conversations remain separate?

*Try to answer each question in one or two sentences.*

## Coding Exercise: Implementing Persistent Memory

This hands-on exercise lets you apply what you've learned. You'll extend the LangGraph application below to incorporate memory and personalize responses.

**Scenario:** You are building a simple dining assistant that remembers each user's dietary preferences and personalizes its responses.
.

### Tasks to Complete

Modify the application below by implementing the following:

1. **Add Memory Persistence**
   - Import `InMemorySaver` from `langgraph.checkpoint.memory`
   - Modify the workflow to compile with this checkpointer

2. **Track User Preferences**
    - Use `InMemorySaver` for state persistence so that the assistant remembers preferences across separate invocations.
    - Pass a unique thread ID in the `config` to ensure conversation isolation for each user.
    - Extend the application state with a `user_memory` dictionary to store dietary preferences (e.g. vegetarian, vegan, gluten-free) and a visit counter.
    - Implement the `remember_preferences` function that detects preferences from incoming messages, updates the stored preferences and visit count, and appends personalized suggestions and welcome-back messages.
    - Connect the `remember_preferences` node to the appropriate edges in the workflow.
    - Compile the workflow with a checkpointer and test it by invoking it multiple times with the same thread ID to verify persistence.

3. **Test Persistence**
   - Invoke the workflow twice using the same `thread_id`
   - Observe how the second invocation uses the stored preference


### Starter Code

Complete the code below to implement the memory persistence features:

In [5]:
from typing import TypedDict, List, Dict, Any, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AIMessage, HumanMessage, AnyMessage
from langgraph.graph.message import add_messages
from operator import add


def add_user_memory(user_memory_1: UserMemory, user_memory_2: UserMemory) -> UserMemory:
    return {
        'diet': user_memory_1.get('diet', []) + user_memory_2.get('diet', []),
        'visits': user_memory_1.get('visits', 0) + user_memory_2.get('visits', 0)
    }


class UserMemory(TypedDict):
    diet: Annotated[List[str], add]
    visits: Annotated[int, add]


class MemoryState(TypedDict):
    messages: Annotated[List[str], add_messages]
    user_id: str
    user_memory: Annotated[UserMemory, add_user_memory]





def greet_user(state: MemoryState) -> dict:
    user_id = state['user_id']
    visits = state['user_memory'].get('visits', 0)
    
    if visits == 0:
        greeting = f"Hello, {user_id}. Welcome!"
    else:
        greeting = f"Welcome back, {user_id}! (Visit #{visits + 1})"
    
    return {
        "messages": [AIMessage(greeting)],
    }


def remember_preferences(state: MemoryState) -> dict:
    messages = state.get('messages')
    
    # 1. Extract preference from the latest HumanMessage
    latest_user_message = [
        msg.content for msg in messages if isinstance(msg, HumanMessage)
    ][-1]
    
    dietary_preferences = None
    if ':' in latest_user_message:
        dietary_preferences = latest_user_message.split(': ')[-1]
    elif latest_user_message.lower().startswith("i'm also"):
        # Extracts 'gluten-free' from "I'm also gluten-free"
        parts = latest_user_message.split()
        if len(parts) > 2:
            dietary_preferences = parts[2]
    elif latest_user_message.lower().startswith("i'm"):
        # Extracts 'vegan' from "I'm vegan and looking for options"
        parts = latest_user_message.split()
        if len(parts) > 1:
            dietary_preferences = parts[1]
    
    if dietary_preferences is None:
        # If no preference is found, return empty dict to make no changes
        return {}

    # 2. Return the partial update
    return {
        'user_memory': {
            'diet': [dietary_preferences],
            # Visits should be tracked by this node for a successful interaction
            'visits': 1,  
        },
        'messages': [AIMessage(f"Got it! I've noted you are {dietary_preferences}.")]
    }

# Build a minimal workflow without persistence
workflow = StateGraph(MemoryState)
workflow.add_node("greet_user", greet_user)
workflow.add_node("remember_preferences", remember_preferences)

workflow.add_edge(START, "greet_user")
workflow.add_edge("greet_user", "remember_preferences")
workflow.add_edge("remember_preferences", END)

app = workflow.compile(checkpointer=InMemorySaver())

# TODO: Implement the following:
# 3. Write a node function remember_preferences(state: MemoryState) -> MemoryState that:
#    - Appends a personalized dish suggestion and a welcome-back message on return visits
# 4. Add this new node to the workflow after the greet node and update the edges accordingly.
# 6. Test your implementation by invoking the workflow multiple times with the same thread_id and observing the persisted state.


### Test Your Implementation

Use the cell below to test your implementation:

In [6]:
# config with a unique thread_id
config = {"configurable": {"thread_id": "user123"}}

# First interaction: user states a dietary preference
state1 = {
    "messages": ["I'm vegan and looking for options"],
    "user_id": "user123",
    "user_memory": {}
}

result1 = app.invoke(state1, config=config)
print("First interaction - User Memory:", result1["user_memory"])
# Expected: {'diet': ['vegan'], 'visits': 1}

print("First interaction - Messages:", result1["messages"])
# Expected to include a vegan dish suggestion

# Second interaction resumes from saved state
saved_state = app.get_state(config).values

# User adds another preference
# 1. Create the new user message
new_message = HumanMessage(content="I'm also gluten-free")

# 2. Invoke with ONLY the new input (messages) to be merged.
# The previous state is loaded via config.
result2 = app.invoke(
    {
        "messages": [new_message]
    }, 
    config=config
)
# Show the latest messages and memory
print("Second interaction - Messages (last 2):", result2["messages"][-2:])
# Expected: a gluten-free suggestion and a welcome-back message

print("Second interaction - User Memory:", result2["user_memory"])
# Expected: {'diet': ['vegan', 'gluten-free'], 'visits': 2}

First interaction - User Memory: {'diet': ['vegan'], 'visits': 1}
First interaction - Messages: [HumanMessage(content="I'm vegan and looking for options", additional_kwargs={}, response_metadata={}, id='2155fd35-7304-4081-a961-1ee72a205984'), AIMessage(content='Hello, user123. Welcome!', additional_kwargs={}, response_metadata={}, id='eda24e4d-b42b-4b30-b38f-ce8425876a09'), AIMessage(content="Got it! I've noted you are vegan.", additional_kwargs={}, response_metadata={}, id='abc4d29a-e0a3-4b8f-9882-e85759c2b967')]
Second interaction - Messages (last 2): [AIMessage(content='Welcome back, user123! (Visit #2)', additional_kwargs={}, response_metadata={}, id='f3ea8210-0ba8-4ea3-9863-d6c6a42c7e0e'), AIMessage(content="Got it! I've noted you are gluten-free.", additional_kwargs={}, response_metadata={}, id='d4f498c9-6f81-46aa-af58-316cf85f4994')]
Second interaction - User Memory: {'diet': ['vegan', 'gluten-free'], 'visits': 2}


### Reflection

After completing the practical exercise, jot down a few sentences about what you found challenging and how you resolved it. Reflect on how memory persistence and thread isolation could be useful in real-world conversational systems.

